# Sample Final Project Inspiration: CISA KEV Mini Tracker

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lolusername/CST4714_DB_admin/blob/main/week_13/sample_final_project_cisa_kev_tracker.ipynb)

This is not a required template.
It is a small example of how a final project can be organized.

The project idea: build a small database that tracks known exploited vulnerabilities from CISA's public KEV catalog.

Final project due date: May 26, 2026.

## What this sample demonstrates

A good final project does not need to be huge.
It needs to show evidence:

- clear project purpose
- small data model
- seed data
- useful queries
- one index decision
- one admin or reliability decision
- a short explanation of tradeoffs

This notebook uses pandas only so you can understand the project before adding a cloud database.

In [ ]:
import pandas as pd

## 1. Project purpose

Project title: CISA KEV Mini Tracker

Problem statement:
Security teams need a simple way to review known exploited vulnerabilities by vendor, product, due date, and ransomware campaign status.

Possible users:
- security analyst
- IT administrator
- vulnerability management team
- database administrator preparing a reporting database

In [ ]:
csv_url = "https://raw.githubusercontent.com/lolusername/CST4714_DB_admin/main/week_12/sample_cisa_kev_vulnerabilities.csv"

df = pd.read_csv(csv_url)

print("Rows and columns:", df.shape)
df.head()

## 2. Data model idea

This data can fit either platform.

Supabase/Postgres version:
- one table named `vulnerabilities`
- `cve_id` as primary key
- useful indexes on `vendor_project`, `date_added`, and `known_ransomware_campaign_use`

MongoDB/Atlas version:
- one collection named `vulnerabilities`
- one document per CVE
- useful indexes on `cve_id`, `vendor_project`, and `known_ransomware_campaign_use`

In [ ]:
# Rename columns to names that are easier to use in SQL or MongoDB.

clean_df = df.rename(columns={
    "cveID": "cve_id",
    "vendorProject": "vendor_project",
    "vulnerabilityName": "vulnerability_name",
    "dateAdded": "date_added",
    "shortDescription": "short_description",
    "requiredAction": "required_action",
    "dueDate": "due_date",
    "knownRansomwareCampaignUse": "known_ransomware_campaign_use",
})

clean_df = clean_df.head(50).copy()

clean_df[["cve_id", "vendor_project", "product", "date_added", "known_ransomware_campaign_use"]].head()

## 3. Supabase/Postgres schema sketch

This is the kind of SQL file a student could include in the final project evidence.

In [ ]:
postgres_schema = """
create table vulnerabilities (
    cve_id text primary key,
    vendor_project text,
    product text,
    vulnerability_name text,
    date_added date,
    due_date date,
    known_ransomware_campaign_use text,
    short_description text,
    required_action text,
    notes text,
    cwes text
);

create index idx_vulnerabilities_vendor on vulnerabilities (vendor_project);
create index idx_vulnerabilities_date_added on vulnerabilities (date_added);
create index idx_vulnerabilities_ransomware on vulnerabilities (known_ransomware_campaign_use);
"""

print(postgres_schema)

## 4. MongoDB document sketch

This is the kind of document model a student could include in the final project evidence.

In [ ]:
mongo_documents = clean_df.to_dict(orient="records")

mongo_documents[0]

## 5. Four useful project questions

These are the kinds of questions the final project database should answer.

In [ ]:
# Question 1: Which vendors appear most often in the sample?

clean_df["vendor_project"].value_counts().head(10)

In [ ]:
# Question 2: Which vulnerabilities are associated with known ransomware campaign use?

clean_df[clean_df["known_ransomware_campaign_use"].str.lower() == "known"][[
    "cve_id", "vendor_project", "product", "known_ransomware_campaign_use"
]].head(10)

In [ ]:
# Question 3: What are the newest vulnerabilities in the sample?

clean_df.sort_values("date_added", ascending=False)[[
    "cve_id", "vendor_project", "product", "date_added"
]].head(10)

In [ ]:
# Question 4: Search for vulnerabilities involving a product or vendor keyword.

keyword = "Microsoft"

clean_df[
    clean_df["vendor_project"].str.contains(keyword, case=False, na=False)
    | clean_df["product"].str.contains(keyword, case=False, na=False)
][["cve_id", "vendor_project", "product", "date_added"]].head(10)

## 6. Query evidence examples

A student would translate the project questions into database queries.

In [ ]:
postgres_query_examples = """
-- Question 1: vendors with the most vulnerabilities
select vendor_project, count(*) as vulnerability_count
from vulnerabilities
group by vendor_project
order by vulnerability_count desc
limit 10;

-- Question 2: ransomware-related records
select cve_id, vendor_project, product
from vulnerabilities
where lower(known_ransomware_campaign_use) = 'known';

-- Question 3: newest records
select cve_id, vendor_project, product, date_added
from vulnerabilities
order by date_added desc
limit 10;

-- Question 4: keyword search
select cve_id, vendor_project, product
from vulnerabilities
where vendor_project ilike '%Microsoft%'
   or product ilike '%Microsoft%';
"""

print(postgres_query_examples)

In [ ]:
mongodb_query_examples = """
// Question 1: vendors with the most vulnerabilities
db.vulnerabilities.aggregate([
  { $group: { _id: "$vendor_project", vulnerability_count: { $sum: 1 } } },
  { $sort: { vulnerability_count: -1 } },
  { $limit: 10 }
])

// Question 2: ransomware-related records
db.vulnerabilities.find({ known_ransomware_campaign_use: "Known" })

// Question 3: newest records
db.vulnerabilities.find().sort({ date_added: -1 }).limit(10)

// Question 4: keyword search with a simple regex
db.vulnerabilities.find({
  $or: [
    { vendor_project: /Microsoft/i },
    { product: /Microsoft/i }
  ]
})
"""

print(mongodb_query_examples)

## 7. Admin and reliability evidence

Example admin choices:

- Index `vendor_project` because the dashboard often filters by vendor.
- Index `known_ransomware_campaign_use` because security users may prioritize those records.
- Use `cve_id` as the primary key or unique identifier.
- Keep a copy of the original CSV source and a clean transformed version.
- Restore verification: after restore, check row count, one vendor query, one ransomware query, and a sample CVE lookup.

In [ ]:
restore_checklist = [
    "Confirm vulnerabilities table or collection exists",
    "Confirm record count is close to expected seed data count",
    "Look up one known CVE by cve_id",
    "Run vendor count query",
    "Run ransomware status query",
    "Confirm important indexes still exist",
]

for item in restore_checklist:
    print("-", item)

## 8. What makes this a reasonable final project?

This project is small but complete.

It has:

- a real public dataset
- a clear user need
- a simple schema or collection model
- useful queries
- index decisions
- admin and restore evidence

A student could make this better by adding a small dashboard, more data cleaning, Atlas Search, RLS policies, or a scheduled refresh plan.